# Modelos y resultados

*7 modelos de regresión evaluados en 2017, un año que no vieron al entrenar*

## Entrenamiento de los modelos

Aquí se entrenan los 7 modelos con el mismo procedimiento de `model/train_model.py`: los datos se
separan en el tiempo (entrenamiento 2015–2016 y prueba 2017), cada modelo ajusta sus hiperparámetros
con Grid Search y validación cruzada temporal, y se evalúa una sola vez en 2017. El código está
visible y comentado; tarda alrededor de medio minuto.

### 1. Datos del modelo

In [ ]:
import time
import warnings

import numpy as np
import pandas as pd
from sklearn.ensemble import RandomForestRegressor
from sklearn.linear_model import Lasso, Ridge
from sklearn.model_selection import GridSearchCV, TimeSeriesSplit
from sklearn.neighbors import KNeighborsRegressor
from sklearn.pipeline import Pipeline
from sklearn.svm import SVR
from sklearn.tree import DecisionTreeRegressor
from xgboost import XGBRegressor

from common import load_artifact, load_mensual, tabla, usd
from model.evaluate import metricas
from model.features import FEATURES, SEED, TARGET, a_ventas, agregar_variables, preprocesador

# Avisos de convergencia de algunos modelos: no cambian el resultado
warnings.filterwarnings("ignore", category=UserWarning)

# Tabla mensual del ETL: una fila por categoría × región × mes (12 series × 48 meses)
panel = load_mensual()

# Agrega las variables del modelo (todas usan solo meses anteriores, model/features.py) y el
# objetivo: el índice = ventas del mes ÷ promedio de los 12 meses anteriores.
# Se descartan las filas sin 12 meses de historia (todo 2014).
datos = agregar_variables(panel).dropna(subset=FEATURES + [TARGET])

# División en el tiempo, nunca al azar: se entrena con el pasado y se prueba con el futuro
train = datos[datos.Mes.dt.year < 2017]   # 2015–2016
test = datos[datos.Mes.dt.year == 2017]   # 2017: se usa una sola vez, para evaluar
X_train, y_train = train[FEATURES], train[TARGET]

print(f"Entrenamiento: {len(train)} filas ({train.Mes.min():%Y-%m} a {train.Mes.max():%Y-%m})")
print(f"Prueba:        {len(test)} filas ({test.Mes.min():%Y-%m} a {test.Mes.max():%Y-%m})")
print("Variables:    ", ", ".join(FEATURES))
print("Objetivo:     ", TARGET)

### 2. Modelos, rejillas de hiperparámetros y validación cruzada

In [ ]:
# Cada modelo con la rejilla de hiperparámetros que prueba Grid Search (las mismas de model/train_model.py)
MODELOS = {
    # Regresión lineal con penalización L2: alpha controla cuánto se encogen los coeficientes
    "Ridge": (Ridge(), {"alpha": [0.01, 0.1, 1, 10, 100]}),
    # Penalización L1: puede dejar coeficientes en cero, es decir, descartar variables
    "Lasso": (Lasso(max_iter=20000), {"alpha": [1e-4, 1e-3, 1e-2, 1e-1]}),
    # Promedia los k meses más parecidos; "distance" pesa más a los vecinos más cercanos
    "KNN": (KNeighborsRegressor(), {"n_neighbors": [3, 5, 10, 15, 20, 30], "weights": ["uniform", "distance"]}),
    # Un solo árbol: la profundidad y el tamaño mínimo de hoja limitan su complejidad
    "Árbol de decisión": (DecisionTreeRegressor(random_state=SEED),
                          {"max_depth": [2, 3, 4, 6], "min_samples_leaf": [1, 5, 10, 20]}),
    # 300 árboles entrenados con muestras distintas; max_features=0.5 hace los árboles más diferentes
    "Random Forest": (RandomForestRegressor(n_estimators=300, random_state=SEED, n_jobs=1),
                      {"max_depth": [3, 5, None], "min_samples_leaf": [1, 3, 5, 10], "max_features": [1.0, 0.5]}),
    # 300 árboles en secuencia, cada uno corrige el error de los anteriores; learning_rate frena cada paso
    "XGBoost": (XGBRegressor(n_estimators=300, random_state=SEED, n_jobs=1, tree_method="hist"),
                {"max_depth": [2, 3, 4], "learning_rate": [0.03, 0.1], "subsample": [0.8, 1.0],
                 "min_child_weight": [1, 5]}),
    # Ajusta una función con un margen de tolerancia epsilon; C castiga los errores fuera del margen
    "SVR": (SVR(), {"C": [0.1, 1, 10], "epsilon": [0.05, 0.1, 0.2], "gamma": ["scale", 0.05]}),
}

# Validación cruzada temporal: 4 pliegues; cada uno entrena con todos los meses anteriores y valida
# con los 3 meses siguientes (3 meses × 12 series = 36 filas). Nunca valida con meses del pasado.
CV = TimeSeriesSplit(n_splits=4, test_size=3 * 12)

# Resumen: cuántas combinaciones prueba cada modelo
tabla(pd.DataFrame([{
    "Modelo": nombre,
    "Combinaciones": int(np.prod([len(v) for v in rejilla.values()])),
    "Rejilla": "; ".join(f"{k} ∈ {v}" for k, v in rejilla.items()),
} for nombre, (_, rejilla) in MODELOS.items()]))

### 3. Entrenamiento con Grid Search y evaluación en 2017

In [ ]:
buscadores, filas = {}, []
for nombre, (estimador, rejilla) in MODELOS.items():
    # Pipeline: el preprocesamiento (one-hot para categoría y región, estandarización para las
    # numéricas) se ajusta dentro de cada pliegue solo con sus datos de entrenamiento
    pipe = Pipeline([("pre", preprocesador()), ("modelo", estimador)])

    # Grid Search: entrena cada combinación de la rejilla en los 4 pliegues y se queda con la de
    # menor MAE promedio en validación. Al final reentrena esa combinación con todo 2015–2016.
    busqueda = GridSearchCV(pipe, {f"modelo__{k}": v for k, v in rejilla.items()}, cv=CV,
                            scoring="neg_mean_absolute_error", n_jobs=-1)
    t0 = time.perf_counter()
    busqueda.fit(X_train, y_train)  # aquí se entrena el modelo
    segundos = time.perf_counter() - t0
    buscadores[nombre] = busqueda

    # Evaluación en 2017, que el modelo no vio: el índice predicho se convierte a dólares
    pred = a_ventas(busqueda.predict(test[FEATURES]), test)
    m = metricas(test.Sales, pred)
    filas.append({
        "Modelo": nombre,
        "Error de validación (MAE del índice)": f"{-busqueda.best_score_:.4f}",
        "MAE 2017": usd(m["MAE"]), "RMSE 2017": usd(m["RMSE"]), "R² 2017": f"{m['R2']:.2f}",
        "Mejores hiperparámetros": ", ".join(f"{k.split('__')[1]}={v}" for k, v in busqueda.best_params_.items()),
        "Segundos": f"{segundos:.1f}",
    })

# Referencia para comparar: vender lo mismo que el mismo mes del año anterior (lag_12)
ref = metricas(test.Sales, test.lag_12)
filas.append({"Modelo": "Referencia (año anterior)", "Error de validación (MAE del índice)": "—",
              "MAE 2017": usd(ref["MAE"]), "RMSE 2017": usd(ref["RMSE"]), "R² 2017": f"{ref['R2']:.2f}",
              "Mejores hiperparámetros": "—", "Segundos": "—"})
tabla(pd.DataFrame(filas))

### 4. Elección del mejor modelo

In [ ]:
# El mejor modelo es el de menor error de validación cruzada. La prueba de 2017 no interviene en la
# elección: así sigue siendo una medida independiente del error.
ganador = min(buscadores, key=lambda n: -buscadores[n].best_score_)
mae = metricas(test.Sales, a_ventas(buscadores[ganador].predict(test[FEATURES]), test))["MAE"]
print(f"Mejor modelo: {ganador}")
print("Hiperparámetros:", {k.split("__")[1]: v for k, v in buscadores[ganador].best_params_.items()})
print(f"MAE en 2017: {usd(mae)} (referencia: {usd(ref['MAE'])})")

# Comprobación: este entrenamiento debe llegar al mismo resultado que model/train_model.py, que es
# el que guardó en model/artifacts/ los resultados que usan las secciones siguientes
guardado = load_artifact("resumen.json")
if ganador == guardado["mejor_modelo"] and abs(mae - guardado["prueba"]["MAE"]) < 1:
    print("\nCoincide con los resultados guardados en model/artifacts/.")
else:
    print(f"\nDifiere de lo guardado ({guardado['mejor_modelo']}, MAE {usd(guardado['prueba']['MAE'])}): "
          "revise que las versiones de las librerías sean las de requirements.txt.")

Además de este entrenamiento, `model/train_model.py` reentrena el mejor modelo con 2015–2017, proyecta
2018 mes a mes, calcula la importancia de variables y guarda todo en `model/artifacts/`. Las secciones
siguientes muestran esos resultados.

## Resultados de los modelos

In [ ]:
import json

import pandas as pd
import plotly.graph_objects as go

from common import (BRAND, BRAND_LIGHT, PROY, REAL, REF, REFERENCIA, kpis, load_artifact, md,
                    mejores_por_modelo, show, style_fig, tabla, usd)

# Resultados guardados por model/train_model.py
resumen = load_artifact("resumen.json")
# res: los 7 modelos ordenados por MAE en 2017 · r: incluye la referencia
res = mejores_por_modelo()
r = load_artifact("resultados_modelos.csv")
ref = r[r.modelo == REFERENCIA].iloc[0]
# Mejor modelo y sus métricas frente a la referencia
best = resumen["mejor_modelo"]
p, rf = resumen["prueba"], resumen["referencia"]
# Cuánto menos error tiene el mejor modelo que la referencia, en %
mejora = (1 - p["MAE"] / rf["MAE"]) * 100
# Cifras clave del mejor modelo
kpis([
    ("Mejor modelo", best, "elegido por validación cruzada (Grid Search)"),
    ("MAE en 2017", usd(p["MAE"]), f"{mejora:.0f} % menos error que la referencia ({usd(rf['MAE'])})"),
    ("Error en el total mensual", f"{p['total_MAE_pct']:.1f} %",
     f"referencia: {rf['total_MAE_pct']:.1f} % · R² {p['total_R2']:.2f}"),
    ("Desvío del total 2017", f"{p['total_desvio_total_pct']:+.1f} %",
     f"referencia: {rf['total_desvio_total_pct']:+.1f} %"),
])

## Real vs. proyectado: ventas totales de la empresa por mes

In [ ]:
# Real vs. predicho de cada modelo para cada serie y mes, en entrenamiento y prueba
pred = load_artifact("predicciones.csv")
modelos = list(res.modelo) + [REFERENCIA]
# Total de la empresa por mes (suma de las 12 series)
t = pred.groupby(["Mes", "conjunto"], as_index=False)[["Sales"] + modelos].sum()
# Inicio y fin del periodo de prueba, para sombrearlo
ini, fin = t[t.conjunto == "Prueba"].Mes.min(), t.Mes.max()

fig = go.Figure()
fig.add_vrect(x0=ini, x1=fin, fillcolor=BRAND_LIGHT, opacity=0.7, line_width=0, layer="below",
              annotation_text="Prueba (2017)", annotation_position="top left")
# Ventas reales
fig.add_trace(go.Scatter(x=t.Mes, y=t.Sales, name="Real", mode="lines+markers",
                         line=dict(color=REAL, width=2), marker_size=5))
# Dos trazas por modelo (entrenamiento y prueba); el menú muestra las del modelo elegido
for m in modelos:
    for conj, dash in [("Entrenamiento", "dot"), ("Prueba", "solid")]:
        s = t[t.conjunto == conj]
        fig.add_trace(go.Scatter(x=s.Mes, y=s[m], name=f"Proyección ({conj.lower()})", mode="lines",
                                 line=dict(color=PROY, width=2.5, dash=dash), visible=(m == best)))
# Menú para elegir el modelo: muestra la serie real y las dos trazas de ese modelo
botones = [dict(label=m, method="update",
                args=[{"visible": [True] + [x == m for x in modelos for _ in range(2)]}]) for m in modelos]
fig.update_layout(updatemenus=[dict(buttons=botones, active=modelos.index(best), x=0, xanchor="left",
                                    y=1.18, yanchor="top")])
fig.update_yaxes(title="Ventas del mes (USD)", tickprefix="$", tickformat=",.0f")
style_fig(fig, 400)
fig.update_layout(margin=dict(t=60))
show(fig)

Cada punto es la suma de las 12 series del mes. La zona sombreada es 2017: el modelo no vio esos
meses al entrenar, cada mes se proyecta con la historia hasta el mes anterior. Elija otro modelo en el
menú para comparar: la referencia (repetir el año anterior) queda por debajo porque no capta el
crecimiento de 2017.

## Resultados por modelo (prueba 2017)

Métricas por categoría × región × mes:

In [ ]:
# Hiperparámetros elegidos, como texto ("alpha=100, ...")
def params(h):
    return ", ".join(f"{k}={v:.3g}" if isinstance(v, float) else f"{k}={v}" for k, v in json.loads(h).items())

# Una fila por modelo con sus métricas en 2017
filas = [{"Modelo": f.modelo, "MAE": usd(f.MAE), "RMSE": usd(f.RMSE), "R²": f"{f.R2:.2f}",
          "Error total mensual": f"{f.total_MAE_pct:.1f} %", "Hiperparámetros": params(f.hiperparametros)}
         for f in res.itertuples()]
# Última fila: la referencia, para comparar
filas.append({"Modelo": REFERENCIA, "MAE": usd(ref.MAE), "RMSE": usd(ref.RMSE), "R²": f"{ref.R2:.2f}",
              "Error total mensual": f"{ref.total_MAE_pct:.1f} %", "Hiperparámetros": "—"})
tabla(pd.DataFrame(filas))

In [ ]:
# Lectura de la tabla escrita con las cifras calculadas
md(f"MAE y RMSE miden el error de cada serie mensual (la venta media por serie y mes en 2017 fue de "
   f"{usd(resumen['ventas_2017'] / 144)}), por eso su R² es bajo: una categoría en una región varía mucho "
   f"de un mes a otro. Al sumar las series, el error del total mensual de la empresa baja a "
   f"{p['total_MAE_pct']:.0f} %. {(res.MAE < ref.MAE).sum()} de {len(res)} modelos superan a la "
   f"referencia en MAE.")

## Ranking de modelos (MAE en 2017)

In [ ]:
import plotly.graph_objects as go

from common import (POS, PROY, REAL, REF, REFERENCIA, kpis, load_artifact, md, mejores_por_modelo,
                    show, style_fig, usd)

# Resultados guardados por model/train_model.py
resumen = load_artifact("resumen.json")
r = load_artifact("resultados_modelos.csv")
ref = r[r.modelo == REFERENCIA].iloc[0]
res = mejores_por_modelo()
best = resumen["mejor_modelo"]
rec, rf = resumen["prueba_recursiva"], resumen["referencia"]

# Barras de MAE de mayor a menor error; el mejor modelo en verde
orden = res.sort_values("MAE", ascending=False)
fig = go.Figure(go.Bar(x=orden.MAE, y=orden.modelo, orientation="h",
                       marker_color=[POS if m == best else REAL for m in orden.modelo],
                       text=[usd(v) for v in orden.MAE], textposition="inside", insidetextanchor="end",
                       textfont=dict(color="white")))
# Línea vertical: error de la referencia
fig.add_vline(x=ref.MAE, line_dash="dash", line_color=REF,
              annotation_text=f"Referencia {usd(ref.MAE)}", annotation_position="top")
fig.update_xaxes(title="MAE en 2017 (USD, menor es mejor)", range=[0, ref.MAE * 1.18])
show(style_fig(fig, 320))

In [ ]:
# Si otro modelo tuvo menos error en 2017, explicar por qué no se eligió
menor, elegido = res.iloc[0], res.set_index("modelo").loc[best]
nota_menor = "" if menor.modelo == best else (
    f" {menor.modelo} tuvo el menor MAE en 2017 ({usd(menor.MAE)}) y no se eligió por dos razones: elegir "
    f"mirando 2017 haría que esa prueba dejara de ser independiente (en validación cruzada su error fue "
    f"{menor.cv_MAE_indice:.3f}, contra {elegido.cv_MAE_indice:.3f} de {best}), y su total del año quedó a "
    f"{menor.total_desvio_total_pct:+.1f} % del real: subestima el año, mientras que {best} quedó a "
    f"{elegido.total_desvio_total_pct:+.1f} %.")
md(f"Todos los modelos con MAE a la izquierda de la línea superan a la referencia. Las diferencias entre "
   f"ellos son pequeñas (de {usd(res.MAE.min())} a {usd(res.MAE.max())}): con 288 filas de entrenamiento, "
   f"la señal disponible es sobre todo la estacionalidad, y todos la capturan. {best} (en verde) se eligió "
   f"por su error de validación cruzada, antes de mirar 2017." + nota_menor)

## Hiperparámetros probados con Grid Search

Rejilla de cada modelo y valor elegido:

In [ ]:
# Formato de un valor de hiperparámetro (None, número o texto)
def fmt(v):
    return "None" if v is None else (f"{v:g}" if isinstance(v, float) else str(v))

# Una fila por modelo: rejilla probada y combinación elegida
tabla(pd.DataFrame([{
    "Modelo": f.modelo,
    "Combinaciones": int(f.evaluaciones),
    "Rejilla probada": "; ".join(f"{k} ∈ {{{', '.join(fmt(x) for x in v)}}}"
                                 for k, v in json.loads(f.rejilla).items()),
    "Elegido por validación cruzada": ", ".join(f"{k}={fmt(v)}" for k, v in json.loads(f.hiperparametros).items()),
} for f in res.itertuples()]))

GridSearchCV entrenó cada combinación de la rejilla con TimeSeriesSplit (4 pliegues) y se quedó con
la de menor MAE en validación cruzada. Los hiperparámetros controlan la complejidad: alpha en Ridge y
Lasso, vecinos en KNN, profundidad y tamaño de hoja en los árboles, tasa de aprendizaje en XGBoost y C
y ε en SVR.

## ¿Hay sobreajuste? Error en entrenamiento vs. prueba

In [ ]:
# MAE de cada modelo en entrenamiento y en prueba, como % de la venta media de ese periodo.
# En porcentaje porque 2017 vende más que 2015–2016: en dólares el error de prueba sube aunque el
# modelo no haya empeorado (le pasa incluso a la referencia).
filas = []
for m in modelos:
    fila = {"modelo": m}
    for conj in ("Entrenamiento", "Prueba"):
        s = pred[pred.conjunto == conj]
        fila[conj] = (s.Sales - s[m]).abs().mean() / s.Sales.mean() * 100
    filas.append(fila)
sob = pd.DataFrame(filas).assign(brecha=lambda d: d.Prueba - d.Entrenamiento).sort_values("Prueba")

# Barras: error en entrenamiento (claro) y en prueba (oscuro) de cada modelo
fig = go.Figure([
    go.Bar(x=sob.modelo, y=sob.Entrenamiento, name="Entrenamiento (2015–2016)", marker_color=BRAND_LIGHT,
           marker_line=dict(color=REAL, width=1.5), hovertemplate="%{x}<br>%{y:.1f} %<extra>Entrenamiento</extra>"),
    go.Bar(x=sob.modelo, y=sob.Prueba, name="Prueba (2017)", marker_color=REAL,
           hovertemplate="%{x}<br>%{y:.1f} %<extra>Prueba</extra>"),
])
fig.update_yaxes(title="MAE (% de la venta media por serie)", ticksuffix=" %")
fig.update_xaxes(tickangle=-20)
show(style_fig(fig, 340))

In [ ]:
# Modelos sin la referencia; brecha = error en prueba − error en entrenamiento
sm = sob[sob.modelo != REFERENCIA]
b, rr = sob.set_index("modelo").loc[best], sob.set_index("modelo").loc[REFERENCIA]
# Modelo con más sobreajuste y modelos estables (brecha de 1 punto o menos)
mayor = sm.loc[sm.brecha.idxmax()]
estables = sm[sm.brecha <= 1].modelo.tolist()
# Lectura del gráfico escrita con las cifras calculadas
md(f"Un modelo sobreajustado se equivoca poco con los datos que vio y mucho con los nuevos. Sin brecha "
   f"relevante (1 punto o menos): {', '.join(estables)} y la referencia. Con brecha: los modelos más "
   f"flexibles, que memorizan parte del entrenamiento; la mayor es la de {mayor.modelo} "
   f"({mayor.Entrenamiento:.0f} % → {mayor.Prueba:.0f} %). {best} pasa de {b.Entrenamiento:.0f} % a "
   f"{b.Prueba:.0f} % (+{b.brecha:.0f} puntos): un sobreajuste moderado, pero en 2017 sigue entre los de "
   f"menor error y por debajo de la referencia ({rr.Prueba:.0f} %). La validación cruzada mide el error "
   f"en meses no vistos, así que al elegir el modelo ya tiene en cuenta este efecto.")

## Importancia de variables del mejor modelo

In [ ]:
# Importancia por permutación del mejor modelo (cuánto empeora el MAE al desordenar cada variable)
imp = load_artifact("importancia.csv").sort_values("importancia")
# Barras horizontales con su desviación estándar (30 repeticiones)
fig = go.Figure(go.Bar(x=imp.importancia, y=imp.nombre, orientation="h", marker_color=BRAND,
                       error_x=dict(type="data", array=imp["std"], color=REF, thickness=1)))
fig.update_xaxes(title="Aumento del error al desordenar la variable")
show(style_fig(fig, 300))

In [ ]:
# Variables que no aportan: desordenarlas no empeora el modelo
sin_aporte = imp[imp.importancia <= 0].nombre.tolist()
md("Importancia por permutación en 2017: cuánto aumenta el error si se desordena cada variable. Dominan "
   "el mes del año y las ventas del mismo mes del año anterior, es decir, la estacionalidad que mostró el "
   "EDA. "
   + (f"{', '.join(sin_aporte)} tienen importancia cero o negativa: desordenarlas no empeora el modelo "
      f"(incluso lo mejora un poco por azar), o sea que no aportan información útil en 2017. Categoría y "
      f"región pesan poco porque el índice ya pone cada serie en su propia escala." if sin_aporte else
      "Categoría y región pesan poco porque el índice ya pone cada serie en su propia escala."))

## ¿Cuánto confiar en la proyección de un año completo?

In [ ]:
# Cifras de la prueba a 12 meses: 2017 proyectado solo con datos hasta 2016
kpis([
    ("Desvío del total anual", f"{rec['total_desvio_total_pct']:+.1f} %",
     f"referencia: {rf['total_desvio_total_pct']:+.1f} %"),
    ("Error del total mensual", f"{rec['total_MAE_pct']:.1f} %", f"{usd(rec['total_MAE'])} por mes en promedio"),
    ("R² del total mensual", f"{rec['total_R2']:.2f}", "variación mensual explicada"),
])

### Prueba de la proyección a 12 meses: 2017 proyectado solo con datos hasta 2016

In [ ]:
# Total mensual real y proyectado de 2017 (proyección recursiva, como la de 2018)
rc = load_artifact("prueba_recursiva_2017.csv").groupby("Mes", as_index=False)[["Sales", "Proyeccion"]].sum()
pred = load_artifact("predicciones.csv")
# Referencia en 2017: repetir cada mes de 2016
rp = pred[pred.conjunto == "Prueba"].groupby("Mes", as_index=False)[REFERENCIA].sum()
# Real, proyección y referencia en el mismo gráfico
fig = go.Figure()
fig.add_trace(go.Scatter(x=rc.Mes, y=rc.Sales, name="Real 2017", mode="lines+markers",
                         line=dict(color=REAL, width=2)))
fig.add_trace(go.Scatter(x=rc.Mes, y=rc.Proyeccion, name="Proyección a 12 meses (con datos hasta 2016)",
                         mode="lines", line=dict(color=PROY, width=2.5)))
fig.add_trace(go.Scatter(x=rp.Mes, y=rp[REFERENCIA], name="Referencia (repetir 2016)", mode="lines",
                         line=dict(color=REF, width=2, dash="dash")))
fig.update_yaxes(title="Ventas del mes (USD)", tickprefix="$", tickformat=",.0f")
show(style_fig(fig, 340))

Es la misma forma en que se proyecta 2018: cada mes proyectado se usa como historia del siguiente, sin
conocer ningún dato real del año. El modelo acierta el total del año y la forma estacional; los
errores están en el detalle de cada mes. La referencia subestima todo el año porque repite 2016 sin
el crecimiento.